In [1]:
from Data_Functions import Get_Data, Extract_Initial_Data, Clean_and_preImpute, Impute_df, Add_Luxury_Clusters
from DeepLearning_Models import Prep_Input, SavePredictions
from tensorflow.keras.models import Model
import numpy as np

In [2]:
MLP = False
TabRes = False
CatEmb_MLP = False
FTT = False
save_preds = True
optimize = False
All_Features = True
luxury_transform = 'log_sqrt'   # 'log' for log1p(x), 'log_sqrt' for log1p(sqrt(x))
n_folds = 12
n_repeats = 5
calib = True
calibration_methods = ['no_calib', 'HTS']

In [3]:
df = Get_Data('Data/train.csv')
df_test = Get_Data('Data/test.csv')

df, df_test = Extract_Initial_Data(df, df_test, version = 2)
df0_clean, df0_test_clean = Clean_and_preImpute(df, df_test, verbose = False)

HomePlanet_Features = [col for col in df0_clean.columns if col.startswith('HomePlanet_')]
deck_Features = [col for col in df0_clean.columns if col.startswith('deck_')]
Region_Features = [col for col in df0_clean.columns if col.startswith('Region_')]
Batch_Features = [col for col in df0_clean.columns if col.startswith('Batch_')]
Destination_Features = [col for col in df0_clean.columns if col.startswith('Destination_')]
HomePlanetDeck_Features = [col for col in df0_clean.columns if col.startswith('HomePlanetDeck_')]

cols_to_drop = ['Cabin Number', 'GroupId', 'Age', 'Under_13', 'side_P', 'Last Name', 'Cabin', 'Age_LifeStages_lbs', 'HomePlanet', 'deck', 
                'side', 'VIP'] + HomePlanet_Features + deck_Features + Region_Features + Batch_Features

Selected_Features = ['Spa', 'RoomService', 'VRDeck', 'FoodCourt', 'CryoSleep', 'ShoppingMall', 'side_S', 'Region', 'Batch', 'LastNameLength', 
                     'Age_LifeStages', 'FamilySize', 'HomePlanetDeck', 'Destination']

Numerical_Features = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'ppId', 'GroupSize', 'GroupFamilySize', 'CabinFamilySize', 
                      'CabinSize', 'LastNameLength', 'FamilySize', 'CabinGroupSize', 'FirstNameLength']

add_per_group = False
df_clean, df_test_clean = Impute_df(df0_clean, df0_test_clean, impute_method = 'Impute', columns_to_drop = cols_to_drop, 
                                    luxury_transform = luxury_transform, add_per_group = add_per_group)

X = df_clean.drop(columns = ['Transported']).copy()
y = df_clean['Transported'].copy()

X_test = df_test_clean.copy()

In [4]:
Luxury = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
X['Log_Total_Sq_Spend'] = (X[Luxury].apply(np.expm1).sum(axis = 1)).apply(np.log1p)
X_test['Log_Total_Sq_Spend'] = (X_test[Luxury].apply(np.expm1).sum(axis = 1)).apply(np.log1p)
Numerical_Features = Numerical_Features + ['Log_Total_Sq_Spend']

In [5]:
if add_per_group:
    group_Luxury = [f'group_{feature}' for feature in Luxury]
    Numerical_Features = Numerical_Features + group_Luxury
    X['Log_Total_Sq_GSpend'] = (X[group_Luxury].apply(np.expm1).sum(axis = 1)).apply(np.log1p)
    X_test['Log_Total_Sq_GSpend'] = (X_test[group_Luxury].apply(np.expm1).sum(axis = 1)).apply(np.log1p)
    Numerical_Features = Numerical_Features + ['Log_Total_Sq_GSpend']

In [6]:
X, X_test = Add_Luxury_Clusters(X, X_test)

In [7]:
Binary_Features = ['CryoSleep', 'side_S']
Ordinal_Categorical_Features = ['Batch', 'Region', 'Age_LifeStages']
Nominal_Categorical_Features = ['HomePlanetDeck', 'Destination', 'Lux_Essential', 'Lux_non_Essential']
Ordinal_Levels = {'Batch': 5, 'Region': 5, 'Age_LifeStages': 8}

In [8]:
default_MLP_filename = 'submission_MLP.csv'
default_TabRes_filename = 'submission_TabRes.csv'
default_CatEmb_filename = 'submission_CatEmb.csv'
default_FTTransformer_filename = 'submission_FTT.csv'

default_fit_params = {'epochs': 600, 'batch_size': 64, 'patience': 150, 'monitor': 'val_accuracy', 'verbose': True, 'custom_callback': True}
best_MLP_params = {'hidden_layers': [179], 'h_activation': 'mish', 'momentum': 0.8, 'dropout_rate': 0.2, 'learning_rate': 0.01, 'l2_weight': 1e-4}
best_TabRes_params = {'hidden_layers': [164], 'residual_layers': [140], 'N_Blocks': 3, 'p_activation': 'mish', 'h_activation': 'relu', 'res_activation': 'mish', 
                      'dropout_rate1': 0.4, 'dropout_rate2': 0.2, 'learning_rate': 0.006, 'l2_weight': 3.4e-4, 'momentum': 0.85}
best_CatEmb_params = {'hidden_layers': [47, 99], 'h_activation': 'mish', 'momentum': 0.95, 'd_ratios': [1.0, 1.6, 2.0, 1.0, 2.0, 0.5], 'l2_weight': 0.00032, 
                      'dropout_rate': 0.34, 'learning_rate': 0.003}

transformer_backbone_kwargs0 = {'n_blocks': 3, 'd_block': 12, 'n_heads': 2, 'ffn_d_hidden_multiplier': 1, 'ffn_activation': 'relu', 'tfout_activation': 'mish', 
                                'ffn_dropout': 0.3, 'residual_dropout': 0.3}
mlp_backbone_kwargs0 = {'hidden_layers': [128, 16], 'mlp_activation': 'mish', 'momentum': 0.98, 'mlp_dropout': 0.33, 'l2_weight': 0.00035}

best_transformer_backbone_kwargs = transformer_backbone_kwargs0.copy()
best_transformer_backbone_kwargs['ffn_activation'] = 'pmish'
best_mlp_backbone_kwargs = mlp_backbone_kwargs0.copy()
best_mlp_backbone_kwargs['hidden_layers'] = [56, 16]
best_transformer_backbone_kwargs['d_block'] = 36
best_transformer_backbone_kwargs['residual_dropout'] = 0.05

if best_transformer_backbone_kwargs['ffn_activation'] == 'pmish_fixed':
    best_transformer_backbone_kwargs['alpha'] = 0.25
    best_transformer_backbone_kwargs['beta'] = 1.75

if best_transformer_backbone_kwargs['ffn_activation'] in ['ReGLU', 'GeGLU', 'MiGLU']:
    best_transformer_backbone_kwargs['ffn_d_hidden_multiplier'] = 0.5

In [9]:
MLP_filename = default_MLP_filename
TabRes_filename = default_TabRes_filename
CatEmb_filename = default_CatEmb_filename
FTTransformer_filename = default_FTTransformer_filename

fit_params = default_fit_params.copy()
fit_params['verbose'] = False
MLP_params = best_MLP_params.copy()
TabRes_params = best_TabRes_params.copy()
CatEmb_params = best_CatEmb_params.copy()
transformer_backbone_kwargs = best_transformer_backbone_kwargs.copy()
mlp_backbone_kwargs = best_mlp_backbone_kwargs.copy()

FTT_params = {'output_shape': 1, 'transformer_backbone_kwargs': transformer_backbone_kwargs, 
              'mlp_backbone_kwargs': mlp_backbone_kwargs, 'learning_rate': 0.003}

In [10]:
if All_Features:
    PIn = Prep_Input(Numerical = Numerical_Features, Binary_Categorical = Binary_Features, Ordinal_Categorical = Ordinal_Categorical_Features, 
                     Nominal_Categorical = Nominal_Categorical_Features, Ord_levels = Ordinal_Levels)
else:
    PIn = Prep_Input(Numerical = Numerical_Features, Binary_Categorical = Binary_Features, Ordinal_Categorical = Ordinal_Categorical_Features, 
                     Nominal_Categorical = Nominal_Categorical_Features, Ord_levels = Ordinal_Levels, Selected_Features = Selected_Features)

In [11]:
if MLP:
    from DeepLearning_Models import MLPClassifier
    Xmlp_train = PIn.MLP_Input(X)
    Xmlp_test = PIn.MLP_Input(X_test)
    preprocessor = PIn.Scale_Num_Features()
    
    if optimize:
        from DeepLearning_Models import MLP_Optimize_Parameters
        N_layers = 1
        const_params = {'h_activation': 'mish'}
        best_params = MLP_Optimize_Parameters(X = Xmlp_train, y = y, output_shape = 1, N_layers = N_layers, const_params = const_params, preprocessor = preprocessor)
        print(f'{best_params = }')
    else:
        model = MLPClassifier(input_shape = Xmlp_train.shape[1], output_shape = 1, **MLP_params, preprocessor = preprocessor)
        display(model.model.summary())
        if calib:
            histories, predictions_dict = model.Ensemble_Learning_Calibrated_CV(X = Xmlp_train, y = y, X_test = Xmlp_test, fit_params = fit_params, 
                                                                                n_folds = n_folds, n_repeats = n_repeats, calibration_methods = calibration_methods)
            if save_preds:
                for m in predictions_dict.keys():
                    filename = f'submission_MLP_{m}.csv'
                    y_pred = predictions_dict[m]
                    SavePredictions(y_pred, X_test, filename = filename)
        else:
            histories, preds, y_pred = model.Ensemble_Learning_CV(X = Xmlp_train, y = y, X_test = Xmlp_test, fit_params = fit_params, 
                                                                  n_folds = n_folds, n_repeats = n_repeats, voting_method = 'soft')
            if save_preds:
                SavePredictions(y_pred, X_test, filename = MLP_filename)

In [12]:
if TabRes:
    from DeepLearning_Models import TabResNetClassifier
    Xtb_train = PIn.MLP_Input(X)
    Xtb_test = PIn.MLP_Input(X_test)
    preprocessor = PIn.Scale_Num_Features()

    if optimize:
        from DeepLearning_Models import TabResNet_Optimal_Parameters
        N_layers = 1
        const_params = {'dropout_rate1': 0.4, 
                        'dropout_rate2': 0.2,
                        'learning_rate': 0.006, 
                        'momentum': 0.85}
        best_params = TabResNet_Optimal_Parameters(X = Xtb_train, y = y, output_shape = 1, N_layers = N_layers, const_params = const_params, preprocessor = preprocessor)
    else:
        model = TabResNetClassifier(input_shape = Xtb_train.shape[1], output_shape = 1, **TabRes_params, preprocessor = preprocessor)
        display(model.model.summary())
        if calib:
            histories, predictions_dict = model.Ensemble_Learning_Calibrated_CV(X = Xtb_train, y = y, X_test = Xtb_test, fit_params = fit_params, 
                                                                                n_folds = n_folds, n_repeats = n_repeats, calibration_methods = calibration_methods)
            if save_preds:
                for m in predictions_dict.keys():
                    filename = f'submission_TabRes_{m}_round2.csv'
                    y_pred = predictions_dict[m]
                    SavePredictions(y_pred, X_test, filename = filename)
        else:
            histories, preds, y_pred = model.Ensemble_Learning_CV(X = Xtb_train, y = y, X_test = Xtb_test, fit_params = fit_params, 
                                                                  n_folds = n_folds, n_repeats = n_repeats, voting_method = 'soft')
            if save_preds:
                SavePredictions(y_pred, X_test, filename = TabRes_filename)

In [13]:
if CatEmb_MLP:
    from DeepLearning_Models import CatEmb_MLPClassifier
    input_shape, cat_cardinalities, Xemlp_train = PIn.CatEmb_Input_Params(X)
    Xemlp_test = PIn.CatEmb_Input(X_test)
    preprocessor = PIn.CatEmb_Preprocessor()
    if optimize:
        from DeepLearning_Models import CatEmb_MLP_Optimize_Parameters
        N_layers = 2
        const_params = {'h_activation': 'mish', 'momentum': 0.95, 'dropout_rate': 0.1, 'learning_rate': 0.002}
        best_params = CatEmb_MLP_Optimize_Parameters(X = Xemlp_train, y = y, input_shape = input_shape, cat_cardinalities = cat_cardinalities, output_shape = 1, 
                                                     N_layers = N_layers, const_params = const_params, preprocessor = preprocessor)
    else:
        model = CatEmb_MLPClassifier(input_shape = input_shape, cat_cardinalities = cat_cardinalities, output_shape = 1, 
                                     **CatEmb_params, preprocessor = preprocessor)
        display(model.model.summary())
        if calib:
            histories, predictions_dict = model.Ensemble_Learning_Calibrated_CV(X = Xemlp_train, y = y, X_test = Xemlp_test, fit_params = fit_params, 
                                                                                n_folds = n_folds, n_repeats = n_repeats, calibration_methods = calibration_methods)
            if save_preds:
                for m in predictions_dict.keys():
                    filename = f'submission_CatEmb_{m}.csv'
                    y_pred = predictions_dict[m]
                    SavePredictions(y_pred, X_test, filename = filename)            
        else:
            histories, preds, y_pred = model.Ensemble_Learning_CV(X = Xemlp_train, y = y, X_test = Xemlp_test, fit_params = fit_params, 
                                                                  n_folds = 10, n_repeats = 3, voting_method = 'soft')
            if save_preds:
                SavePredictions(y_pred, X_test, filename = CatEmb_filename)

In [14]:
n_jobs = 8
n_folds = 12
n_repeats = 5

In [15]:
FTT_basename = 'FTT'
nt = 1

use_parallel = True

if FTT:
    from DeepLearning_Models import FTTransformer
    n_cont_features, cat_cardinalities, X_ftt_train = PIn.FTT_Input_Params(X)
    X_ftt_test = PIn.FTT_Input(X_test)
    preprocessor = PIn.FTT_Preprocessor()
    FTT_params['n_cont_features'] = n_cont_features
    FTT_params['cat_cardinalities'] = cat_cardinalities

    model = FTTransformer(name='FTTransformer', **FTT_params, preprocessor=preprocessor)
    display(model.model.summary())
    for layer in model.model.layers:
        if isinstance(layer, Model):
            print(f"Submodel: {layer.name}")
            display(layer.summary())

    if use_parallel:
        histories, predictions_dict = model.parallel_calibrated_cv(X = X_ftt_train, y = y, X_test = X_ftt_test, fit_params = fit_params, n_folds = n_folds, 
                                                                   n_repeats = n_repeats, calibration_methods = calibration_methods, n_jobs = n_jobs)
    else:
        histories, predictions_dict = model.Ensemble_Learning_Calibrated_CV(X = X_ftt_train, y = y, X_test = X_ftt_test, fit_params = fit_params, n_folds = n_folds, 
                                                                            n_repeats = n_repeats, calibration_methods = calibration_methods)

    if save_preds:
        for m in predictions_dict.keys():
            FTTransformer_filename = f'submission_{FTT_basename}_{m}_test{nt}.csv'
            y_pred = predictions_dict[m]
            SavePredictions(y_pred, X_test, filename=FTTransformer_filename)